In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error, root_mean_squared_error, precision_recall_fscore_support
from xgboost import XGBClassifier, XGBRegressor
from sklearn.ensemble import IsolationForest, RandomForestRegressor
import joblib
import os

In [17]:
# Load processed features
df = pd.read_parquet("../data/processed/xjtu_sy_features.parquet")

# Define target labels
# Binary classification: Let's label files where RUL_Percentage < 0.2 as Degraded (1), else Normal (0)
df["Binary_Anomaly"] = (df["RUL_Percentage"] < 0.2).astype(int)

In [18]:
# Features to drop from training matrices
drop_cols = ["Bearing", "FileIndex", "TotalLifetime", "RUL_Percentage", "Binary_Anomaly"]
X = df.drop(columns=drop_cols)

# --- TASK 1: Binary Classification / Anomaly Detection ---
y_class = df["Binary_Anomaly"]
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_class, test_size=0.2, random_state=42, shuffle=True)

In [24]:
xgb_clf = XGBClassifier(n_estimators=100, random_state=42)
xgb_clf.fit(X_train_c, y_train_c)
y_pred_c = xgb_clf.predict(X_test_c)

print("\n--- XGBoost Classifier Metrics ---")
print(classification_report(y_test_c, y_pred_c))


--- XGBoost Classifier Metrics ---
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      1465
           1       0.93      0.92      0.92       379

    accuracy                           0.97      1844
   macro avg       0.96      0.95      0.95      1844
weighted avg       0.97      0.97      0.97      1844



In [20]:
# 2. Isolation Forest (Unsupervised Anomaly Detection)
# Train primarily on normal data (where y_train_c == 0) to learn normal behavior
X_normal_train = X_train_c[y_train_c == 0]
iso_forest = IsolationForest(contamination=0.2, random_state=42)
iso_forest.fit(X_normal_train)

# Isolation Forest predicts 1 for normal and -1 for anomaly. Convert to 0 (normal) and 1 (anomaly).
iso_preds_raw = iso_forest.predict(X_test_c)
y_pred_iso = np.where(iso_preds_raw == -1, 1, 0)

print("\n--- Isolation Forest Metrics ---")
print(classification_report(y_test_c, y_pred_iso))


--- Isolation Forest Metrics ---
              precision    recall  f1-score   support

           0       0.89      0.79      0.84      1465
           1       0.43      0.61      0.51       379

    accuracy                           0.76      1844
   macro avg       0.66      0.70      0.67      1844
weighted avg       0.80      0.76      0.77      1844



In [26]:
# --- TASK 2: RUL Estimation (Regression) ---
y_reg = df["RUL_Percentage"]
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y_reg, test_size=0.2, random_state=42, shuffle=True)

xgb_reg = XGBRegressor(n_estimators=100, random_state=42)
xgb_reg.fit(X_train_r, y_train_r)
y_pred_r = xgb_reg.predict(X_test_r)

mae = mean_absolute_error(y_test_r, y_pred_r)
rmse = root_mean_squared_error(y_test_r, y_pred_r)

print("\n--- XGBoost Regressor Metrics ---")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Square Error (RMSE): {rmse:.4f}")


--- XGBoost Regressor Metrics ---
Mean Absolute Error (MAE): 0.0471
Root Mean Square Error (RMSE): 0.0717


In [22]:
# 2. Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_reg.fit(X_train_r, y_train_r)
y_pred_rf_r = rf_reg.predict(X_test_r)

rf_mae = mean_absolute_error(y_test_r, y_pred_rf_r)
rf_rmse = root_mean_squared_error(y_test_r, y_pred_rf_r)

print("\n--- Random Forest Regressor Metrics ---")
print(f"MAE:  {rf_mae:.4f}")
print(f"RMSE: {rf_rmse:.4f}")


--- Random Forest Regressor Metrics ---
MAE:  0.0403
RMSE: 0.0712


In [28]:
# Save trained models for the Streamlit dashboard
os.makedirs("../saved_models", exist_ok=True)
joblib.dump(xgb_clf, "../saved_models/xgb_classifier.joblib")
joblib.dump(iso_forest, "../saved_models/isolation_forest.joblib")
joblib.dump(xgb_reg, "../saved_models/xgb_regressor.joblib")
joblib.dump(rf_reg, "../saved_models/rf_regressor.joblib")

print("\nAll models successfully trained and saved to saved_models/!")


All models successfully trained and saved to saved_models/!
